# 6-практикалық жұмыс. Регрессиялық модельдерді құру және бағалау
**Датасет:** Airfoil Self-Noise (NASA) · **Өзіндік жұмыс:** Concrete Compressive Strength

Google Colab-та орындалады: `Runtime → Run all`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

AIRFOIL_URL  = "https://raw.githubusercontent.com/inkri/Airfoil_Self_Noise/master/AirfoilSelfNoise.csv"
CONCRETE_URL = "https://raw.githubusercontent.com/georgios-kalomitsinis/Concrete-compressive-strength/master/Concrete_Data.csv"

def load_dataset(url, names):
    """CSV-ны GitHub-тан оқиды; егер сілтеме істемесе - файлды қолмен жүктеуді ұсынады."""
    try:
        df = pd.read_csv(url, sep=None, engine="python")
        try:                       # егер тақырып жолы жоқ болса (бірінші жол - сандар)
            float(df.columns[0])
            df = pd.read_csv(url, sep=None, engine="python", header=None)
        except ValueError:
            pass
    except Exception as e:
        print("GitHub-тан жүктеу сәтсіз:", e)
        from google.colab import files
        up = files.upload()
        df = pd.read_csv(list(up.keys())[0], sep=None, engine="python")
    assert df.shape[1] == len(names), f"Күтілген баған саны {len(names)}, табылғаны {df.shape[1]}"
    df.columns = names
    return df

def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred), "MSE": mse,
            "RMSE": np.sqrt(mse), "R2": r2_score(y_true, y_pred)}

## 1. Деректерді жүктеу және алғашқы талдау (EDA)

In [ ]:
airfoil_cols = ["Frequency", "Angle", "Chord_length", "Velocity", "Displacement", "Sound_pressure"]
df = load_dataset(AIRFOIL_URL, airfoil_cols)
TARGET = "Sound_pressure"
FEATURES = [c for c in df.columns if c != TARGET]

display(df.head())
print("shape:", df.shape)

In [ ]:
df.info()
display(df.describe().T)
print("Бос мәндер:\n", df.isna().sum())
print("Қайталанатын жолдар:", df.duplicated().sum())

In [ ]:
# Қайталанатын жазбалар болса - жоямыз
n_dup = df.duplicated().sum()
if n_dup:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"{n_dup} қайталанатын жол жойылды, жаңа shape:", df.shape)

In [ ]:
# Мақсатты айнымалының әрбір белгіге тәуелділігі
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), FEATURES):
    ax.scatter(df[col], df[TARGET], s=8, alpha=0.4)
    ax.set_xlabel(col); ax.set_ylabel(TARGET); ax.set_title(f"{TARGET} vs {col}")
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

# Корреляциялық матрица
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Корреляциялық матрица"); plt.show()

print(df.corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False))

## 2. Train/Test бөлу (80/20)

In [ ]:
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", X_train.shape, " Test:", X_test.shape)

results = {}   # қорытынды кесте үшін

## 3. Қарапайым сызықтық регрессия

In [ ]:
best_feature = df.corr()[TARGET].drop(TARGET).abs().idxmax()
print("Шу деңгейімен ең тығыз байланысқан белгі:", best_feature)

lr1 = LinearRegression().fit(X_train[[best_feature]], y_train)
pred1 = lr1.predict(X_test[[best_feature]])

print(f"Теңдеу: {TARGET} = {lr1.intercept_:.4f} + ({lr1.coef_[0]:.6f}) * {best_feature}")

plt.figure(figsize=(8, 5))
plt.scatter(X_test[best_feature], y_test, s=10, alpha=0.5, label="Нақты (test)")
xs = np.linspace(X[best_feature].min(), X[best_feature].max(), 100).reshape(-1, 1)
plt.plot(xs, lr1.predict(pd.DataFrame(xs, columns=[best_feature])), color="red", label="Регрессия сызығы")
plt.xlabel(best_feature); plt.ylabel(TARGET); plt.legend(); plt.title("Қарапайым сызықтық регрессия")
plt.show()

c = lr1.coef_[0]
print(f"Түсіндіру: {best_feature} бір бірлікке артқанда {TARGET} орта есеппен {c:.6f} дБ-ге "
      f"{'артады' if c > 0 else 'кемиді'} (қалған факторлар тұрақты деп алынғанда).")

## 4. Модельді бағалау (MAE, MSE, RMSE, R²)

In [ ]:
results["Қарапайым Linear Regression"] = evaluate(y_test, pred1)
m = results["Қарапайым Linear Regression"]
print({k: round(v, 4) for k, v in m.items()})
print(f"\nRMSE = {m['RMSE']:.2f} дБ: модель шу деңгейін орта есеппен шамамен ±{m['RMSE']:.2f} дБ қателікпен болжайды "
      f"(y-тің std = {y.std():.2f} дБ). R² = {m['R2']:.2f}: дисперсияның {m['R2']*100:.0f}%-ын ғана түсіндіреді.")

## 5. Көптік сызықтық регрессия (барлық 5 белгі)

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
pred_lr = lr.predict(X_test)
results["Көптік Linear Regression"] = evaluate(y_test, pred_lr)

print(pd.Series(lr.coef_, index=FEATURES), "\nIntercept:", lr.intercept_)
cmp = pd.DataFrame(results).T
display(cmp)
d = cmp.loc["Қарапайым Linear Regression", "RMSE"] - cmp.loc["Көптік Linear Regression", "RMSE"]
print(f"Қалған белгілерді қосқанда RMSE {d:.2f} дБ-ге өзгерді -> модель сапасы "
      f"{'жақсарды' if d > 0 else 'жақсармады'}.")

## 6. Полиномдық регрессия (degree = 2, 3)
Frequency мәндері өте үлкен болғандықтан, дәреже көтергенде сандық тұрақтылық үшін полином белгілерін `StandardScaler`-мен масштабтаймыз (модельдің мәні өзгермейді).

In [ ]:
poly_models = {}
for deg in [2, 3]:
    pm = make_pipeline(PolynomialFeatures(deg, include_bias=False), StandardScaler(), LinearRegression())
    pm.fit(X_train, y_train)
    poly_models[deg] = pm
    results[f"Polynomial degree={deg}"] = evaluate(y_test, pm.predict(X_test))
    tr = r2_score(y_train, pm.predict(X_train)); te = r2_score(y_test, pm.predict(X_test))
    print(f"degree={deg}: белгі саны={pm[0].n_output_features_}, R² train={tr:.4f}, R² test={te:.4f}")
display(pd.DataFrame(results).T)

**Неліктен полином дәрежесін арттыру Test-тегі сапаны міндетті түрде жақсартпайды?**
Дәреже өскен сайын модель икемдірек болып, Train деректеріне жақсы бейімделеді (Train R² өседі), бірақ шуды да «жаттап» алады - *overfitting* (қайта оқыту). Белгі саны тез өседі, коэффициенттер тұрақсыз болады, ал жаңа (Test) деректерде қате өсуі немесе тоқтап қалуы мүмкін. Сондықтан модель күрделілігі Train/Test немесе кросс-валидация арқылы таңдалады.

## 7. Ridge регрессиясы

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100]

def run_regularized(ModelCls, alphas, Xtr, Xte, ytr, yte, names, **kw):
    rows, coefs = [], {}
    for a in alphas:
        mdl = make_pipeline(StandardScaler(), ModelCls(alpha=a, **kw)).fit(Xtr, ytr)
        coefs[a] = mdl[-1].coef_
        met = evaluate(yte, mdl.predict(Xte))
        cv = -cross_val_score(make_pipeline(StandardScaler(), ModelCls(alpha=a, **kw)),
                              Xtr, ytr, cv=5, scoring="neg_root_mean_squared_error").mean()
        rows.append({"alpha": a, **met, "CV_RMSE(train)": cv})
    return pd.DataFrame(rows).set_index("alpha"), pd.DataFrame(coefs, index=names).T

ridge_tab, ridge_coefs = run_regularized(Ridge, alphas, X_train, X_test, y_train, y_test, FEATURES)
display(ridge_tab)
print("Ridge коэффициенттері (стандартталған белгілер):")
display(ridge_coefs)
ridge_coefs.plot(marker="o", logx=True, figsize=(8, 4), title="Ridge: alpha өзгергендегі коэффициенттер")
plt.ylabel("коэффициент"); plt.show()

best_ridge_alpha = ridge_tab["CV_RMSE(train)"].idxmin()
print("CV бойынша ең жақсы alpha (Ridge):", best_ridge_alpha)
ridge_best = make_pipeline(StandardScaler(), Ridge(alpha=best_ridge_alpha)).fit(X_train, y_train)
results["Ridge"] = evaluate(y_test, ridge_best.predict(X_test))

## 8. Lasso регрессиясы

In [ ]:
lasso_tab, lasso_coefs = run_regularized(Lasso, alphas, X_train, X_test, y_train, y_test, FEATURES, max_iter=10000)
display(lasso_tab)
print("Lasso коэффициенттері (стандартталған белгілер):")
display(lasso_coefs)
lasso_coefs.plot(marker="o", logx=True, figsize=(8, 4), title="Lasso: alpha өзгергендегі коэффициенттер")
plt.ylabel("коэффициент"); plt.show()

best_lasso_alpha = lasso_tab["CV_RMSE(train)"].idxmin()
print("CV бойынша ең жақсы alpha (Lasso):", best_lasso_alpha)
lasso_best = make_pipeline(StandardScaler(), Lasso(alpha=best_lasso_alpha, max_iter=10000)).fit(X_train, y_train)
results["Lasso"] = evaluate(y_test, lasso_best.predict(X_test))

In [ ]:
# Linear / Ridge / Lasso коэффициенттерін салыстыру (бәрі стандартталған белгілерде)
lin_std = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
coef_cmp = pd.DataFrame({
    "Linear": lin_std[-1].coef_,
    f"Ridge (alpha={best_ridge_alpha})": ridge_best[-1].coef_,
    f"Lasso (alpha={best_lasso_alpha})": lasso_best[-1].coef_,
}, index=FEATURES)
display(coef_cmp)

shrink = (coef_cmp.iloc[:, 2].abs() / coef_cmp.iloc[:, 0].abs()).sort_values()
print("Lasso коэффициентті Linear-мен салыстырғанда қаншалықты кішірейтті (кіші = қатты азайды):")
print(shrink.round(3))
print("\nLasso ең қатты азайтқан белгі:", shrink.index[0])

## 9. Қорытынды кесте

In [ ]:
table = pd.DataFrame(results).T[["MAE", "MSE", "RMSE", "R2"]].round(4)
table.index.name = "Модель"
display(table)

## 10. Қорытынды тапсырма: ең жақсы модель

In [ ]:
best_name = table["RMSE"].idxmin()
print("Ең жақсы модель (ең төмен Test RMSE):", best_name)
print(table.loc[best_name])

predictors = {
    "Қарапайым Linear Regression": lambda: lr1.predict(X_test[[best_feature]]),
    "Көптік Linear Regression": lambda: lr.predict(X_test),
    "Polynomial degree=2": lambda: poly_models[2].predict(X_test),
    "Polynomial degree=3": lambda: poly_models[3].predict(X_test),
    "Ridge": lambda: ridge_best.predict(X_test),
    "Lasso": lambda: lasso_best.predict(X_test),
}
y_pred_best = predictors[best_name]()
resid = y_test - y_pred_best

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(y_test, y_pred_best, s=10, alpha=0.5)
lims = [y_test.min(), y_test.max()]
ax[0].plot(lims, lims, "r--"); ax[0].set_xlabel("Actual"); ax[0].set_ylabel("Predicted")
ax[0].set_title(f"Actual vs Predicted - {best_name}")
ax[1].scatter(y_pred_best, resid, s=10, alpha=0.5)
ax[1].axhline(0, color="r", linestyle="--"); ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("Residual (Actual - Predicted)")
ax[1].set_title("Қалдықтар графигі")
plt.tight_layout(); plt.show()

sns.histplot(resid, kde=True); plt.title("Қалдықтардың таралуы"); plt.show()
print(f"Негіздеме: '{best_name}' моделі Test жиынында ең төмен RMSE ({table.loc[best_name,'RMSE']:.3f} дБ) және "
      f"ең жоғары R² ({table.loc[best_name,'R2']:.3f}) көрсетті.")

**Қорытынды (толтыру үшін):** жоғарыдағы кестеге сүйене отырып, ең жақсы модельді, оның артықшылығын және қалдықтар графигінде жүйелі ауытқу бар-жоғын (нүктелер 0 маңында кездейсоқ орналасса - модель жарамды) өз сөзіңізбен жазыңыз.

---
# Өзіндік жұмыс: Concrete Compressive Strength
Мақсат - бетонның қысуға беріктігін (MPa) құрамы мен жасы бойынша болжау.

In [ ]:
concrete_cols = ["Cement", "Slag", "Fly_ash", "Water", "Superplasticizer",
                 "Coarse_agg", "Fine_agg", "Age", "Strength"]
cdf = load_dataset(CONCRETE_URL, concrete_cols)
CT = "Strength"
display(cdf.head())
print("shape:", cdf.shape)
cdf.info()
display(cdf.describe().T)
print("Бос мәндер:", cdf.isna().sum().sum(), "| Қайталанатын жолдар:", cdf.duplicated().sum())

In [ ]:
cdf = cdf.drop_duplicates().reset_index(drop=True)

plt.figure(figsize=(9, 7))
sns.heatmap(cdf.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Concrete: корреляциялық матрица"); plt.show()

cfeat = [c for c in cdf.columns if c != CT]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), cfeat):
    ax.scatter(cdf[col], cdf[CT], s=8, alpha=0.4); ax.set_xlabel(col); ax.set_ylabel(CT)
plt.tight_layout(); plt.show()

In [ ]:
# X және y
Xc = cdf[cfeat]; yc = cdf[CT]
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=RANDOM_STATE)
print("Train:", Xc_tr.shape, "Test:", Xc_te.shape)

cres, cmodels = {}, {}

def fit_eval(name, model):
    model.fit(Xc_tr, yc_tr)
    cmodels[name] = model
    cres[name] = evaluate(yc_te, model.predict(Xc_te))
    cres[name]["R2_train"] = r2_score(yc_tr, model.predict(Xc_tr))

fit_eval("Linear Regression", make_pipeline(StandardScaler(), LinearRegression()))
for deg in [2, 3]:
    fit_eval(f"Polynomial degree={deg}",
             make_pipeline(StandardScaler(), PolynomialFeatures(deg, include_bias=False), StandardScaler(), LinearRegression()))

# Ridge / Lasso: alpha-ны 5-fold CV арқылы таңдау (сызықтық белгілер)
ridge_gs = GridSearchCV(make_pipeline(StandardScaler(), Ridge()),
                        {"ridge__alpha": [0.01, 0.1, 1, 10, 100]}, cv=5, scoring="neg_root_mean_squared_error").fit(Xc_tr, yc_tr)
lasso_gs = GridSearchCV(make_pipeline(StandardScaler(), Lasso(max_iter=20000)),
                        {"lasso__alpha": [0.001, 0.01, 0.1, 1, 10]}, cv=5, scoring="neg_root_mean_squared_error").fit(Xc_tr, yc_tr)
print("Ridge best alpha:", ridge_gs.best_params_, "| Lasso best alpha:", lasso_gs.best_params_)
fit_eval("Ridge", ridge_gs.best_estimator_)
fit_eval("Lasso", lasso_gs.best_estimator_)

# Бетон тәуелділігі сызықтық емес болғандықтан - полиномдық белгілермен реттеу де тексереміз
ridge_p = GridSearchCV(make_pipeline(StandardScaler(), PolynomialFeatures(2, include_bias=False), StandardScaler(), Ridge()),
                       {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]}, cv=5, scoring="neg_root_mean_squared_error").fit(Xc_tr, yc_tr)
lasso_p = GridSearchCV(make_pipeline(StandardScaler(), PolynomialFeatures(2, include_bias=False), StandardScaler(), Lasso(max_iter=50000)),
                       {"lasso__alpha": [0.001, 0.01, 0.1, 1]}, cv=5, scoring="neg_root_mean_squared_error").fit(Xc_tr, yc_tr)
print("Ridge(poly2) best:", ridge_p.best_params_, "| Lasso(poly2) best:", lasso_p.best_params_)
fit_eval("Ridge (poly deg=2)", ridge_p.best_estimator_)
fit_eval("Lasso (poly deg=2)", lasso_p.best_estimator_)

ctable = pd.DataFrame(cres).T[["MAE", "MSE", "RMSE", "R2", "R2_train"]].round(4)
ctable.index.name = "Модель"
display(ctable)

In [ ]:
# Ridge/Lasso (сызықтық) коэффициенттері
cc = pd.DataFrame({
    "Linear": cmodels["Linear Regression"][-1].coef_,
    "Ridge": cmodels["Ridge"][-1].coef_,
    "Lasso": cmodels["Lasso"][-1].coef_,
}, index=cfeat).round(3)
display(cc)
print("Lasso нөлге айналдырған белгілер:", list(cc.index[cc["Lasso"] == 0]) or "жоқ")

In [ ]:
cbest = ctable["RMSE"].idxmin()
print("Ең жақсы модель:", cbest); print(ctable.loc[cbest])
yp = cmodels[cbest].predict(Xc_te); rs = yc_te - yp

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(yc_te, yp, s=12, alpha=0.6)
l = [yc_te.min(), yc_te.max()]; ax[0].plot(l, l, "r--")
ax[0].set_xlabel("Actual (MPa)"); ax[0].set_ylabel("Predicted (MPa)"); ax[0].set_title(f"Actual vs Predicted - {cbest}")
ax[1].scatter(yp, rs, s=12, alpha=0.6); ax[1].axhline(0, color="r", linestyle="--")
ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("Residual"); ax[1].set_title("Қалдықтар графигі")
plt.tight_layout(); plt.show()

lin_r2 = ctable.loc["Linear Regression", "R2"]
print(f"Linear Regression R²={lin_r2:.3f}; ең жақсы модель ({cbest}) R²={ctable.loc[cbest,'R2']:.3f}, "
      f"RMSE={ctable.loc[cbest,'RMSE']:.2f} MPa.")
print("Train R² мен Test R² арасындағы үлкен айырма - overfitting белгісі (кестедегі R2_train пен R2 бағандарын салыстырыңыз).")

**Қорытынды (толтыру үшін):** бетон беріктігінің құрамға тәуелділігі сызықтық емес, сондықтан полиномдық белгілер сызықтық модельден жақсы нәтиже беруі керек; Ridge/Lasso күрделі (полиномдық) модельде overfitting-ті азайтуға көмектеседі. Кестедегі нәтижелерге сүйеніп, ең жақсы модельді өз сөзіңізбен негіздеңіз.